In [159]:
import pandas as pd
import torch
from pylate import models, retrieve, indexes
from tqdm.auto import tqdm

In [160]:
# Load the modernColbert
device = "mps" if torch.backends.mps.is_available() else "cpu"
model = models.ColBERT(model_name_or_path="lightonai/colbertv2.0", device=device)

In [161]:
# Load the existing Voyager index
index = indexes.Voyager(
    index_folder="../data/processed/pylate_index",
    index_name="esci_data_index",
    override=False
)
index.ef_search = 128
retriever = retrieve.ColBERT(index=index)

In [162]:
import sqlite3
from sqlitedict import SqliteDict

# Load the Amazon ESCI dataset
test_df = pd.read_parquet("../data/raw/shopping_queries_dataset_examples.parquet",
                         columns=['query_id', 'query', 'product_id', 'esci_label'])

# Get the pids ACTUALLY in the Voyager index via its own metadata
# document_ids_to_embeddings has one key per indexed document (not per token)
doc_map = SqliteDict(
    "../data/processed/pylate_index/esci_data_index/document_ids_to_embeddings.sqlite",
    outer_stack=False
)
indexed_pids = set(int(k) for k in doc_map.keys())
doc_map.close()

# Look up their ASINs from SQLite
conn = sqlite3.connect("../data/processed/products_dataset.db")
all_products = pd.read_sql("SELECT pid, product_id FROM products", conn)
conn.close()

indexed_asins = set(
    all_products[all_products['pid'].isin(indexed_pids)]['product_id']
)

print(f"Voyager token vectors:   {index.index.num_elements}")
print(f"Actual indexed docs:     {len(indexed_pids)}")
print(f"Matching ASINs:          {len(indexed_asins)}")

# Filter ground truth to only products the retriever can actually return
test_df_filtered = test_df[test_df['product_id'].isin(indexed_asins)]

# Only keep queries that have at least one annotated product in the index
valid_query_ids = test_df_filtered['query_id'].unique()
print(f"Queries with indexed ground truth: {len(valid_query_ids)}")

# Sample 100 unique queries from the filtered set
eval_queries = (
    test_df_filtered
    .drop_duplicates('query_id')
    .sample(100, random_state=42)
)
print(f"Sampled eval queries:    {len(eval_queries)}")

Voyager token vectors:   1430662
Actual indexed docs:     20000
Matching ASINs:          20000
Queries with indexed ground truth: 2064
Sampled eval queries:    100


In [163]:
eval_queries['esci_label'].unique()

<ArrowStringArray>
['S', 'E', 'C', 'I']
Length: 4, dtype: str

In [164]:
eval_queries['esci_label'].value_counts()

esci_label
E    58
S    26
I     9
C     7
Name: count, dtype: int64

In [165]:
# Test a single query before running the whole loop
try:
    test_emb = model.encode(["test"], is_query=True)
    test_res = retriever.retrieve(test_emb, k=10)
    print("Search is working!")
except Exception as e:
    print(f" Search still failing: {e}")

Retrieving documents (bs=50): 100%|██████████| 1/1 [00:00<00:00,  3.60it/s]

Search is working!


In [166]:
# Retrieval and label extraction
conn = sqlite3.connect("../data/processed/products_dataset.db")
cursor = conn.cursor()

# Create a mapping for the ESCI shorthand codes
CODE_TO_NAME = {
    "E": "exact",
    "S": "substitute",
    "C": "complement",
    "I": "irrelevant"
}

K = 5
evaluation_results = []

for _, row in tqdm(eval_queries.iterrows(), total=len(eval_queries)):
    q_id = row['query_id']
    q_text = row['query']

    # Get Top-K PIDs from ModernColBERT + Voyager
    query_emb = model.encode([q_text], is_query=True, batch_size=1, show_progress_bar=False)
    search_hits = retriever.retrieve(query_emb, k=K)[0]
    retrieved_pids = [hit['id'] for hit in search_hits]

    # Map integer pid → ASIN + title via SQLite
    placeholders = ",".join("?" * len(retrieved_pids))
    cursor.execute(
        f"SELECT pid, product_id, product_title FROM products WHERE pid IN ({placeholders})",
        retrieved_pids
    )
    pid_rows = {r[0]: (r[1], r[2]) for r in cursor.fetchall()}  # pid -> (asin, title)

    # Ground truth labels for this query (only indexed products)
    query_ground_truth = test_df_filtered[test_df_filtered['query_id'] == q_id]

    hits_with_labels = []
    for rank, pid in enumerate(retrieved_pids):
        asin, title = pid_rows.get(pid, (None, "Title Not Found"))
        if asin is not None:
            match = query_ground_truth[query_ground_truth['product_id'] == asin]
            if not match.empty:
                raw_code = match.iloc[0]['esci_label']
                label = CODE_TO_NAME.get(raw_code, "irrelevant")
            else:
                label = "irrelevant"
        else:
            label = "irrelevant"

        hits_with_labels.append({
            "rank": rank + 1,
            "product_id": asin,
            "title": title,
            "label": label
        })

    evaluation_results.append({
        "query": q_text,
        "hits": hits_with_labels
    })

conn.close()

  0%|          | 0/100 [00:00<?, ?it/s]

Retrieving documents (bs=50): 100%|██████████| 1/1 [00:00<00:00, 11.17it/s]


In [167]:
# Example output for the first query
first_query = evaluation_results[2]
print(f"Query: {first_query['query']}")
for hit in first_query['hits']:
    print(f"Rank {hit['rank']}: PID {hit['product_id']} Product: {hit['title']}: -> Label: {hit['label']}")

Query: 2 in 1 shorts men bonim
Rank 1: PID B08CCVGLC8 Product: bonim Mens 2 in 1 Running Shorts, Quick Dry Workout Athletic Shorts with Pocket & Towel Loop Black XX-Large: -> Label: exact
Rank 2: PID B07W97TKJH Product: MECH-ENG Men's 2 in 1 Shorts Workout Running Training Gym 7" Short with Towel Loop(Black L/Tag 2XL): -> Label: substitute
Rank 3: PID B07WVQ93GM Product: Pinkbomb Men's 2 in 1 Running Shorts Gym Workout Quick Dry Mens Shorts with Phone Pocket (Black, Large): -> Label: substitute
Rank 4: PID B07PXR6QFJ Product: BALEAF Men's 2 in 1 Running Athletic Shorts 5" Quick Dry Workout Shorts with Liner Zipper Pocket Grey/Black Size M: -> Label: substitute
Rank 5: PID B07QFTDZ11 Product: OuBER Men's 2-in-1 Running Shorts 7" Workout Training Jersey Short (Black,L): -> Label: substitute


In [168]:
import numpy as np

# Relevance scores for hits (stored as full words by the retrieval loop)
RELEVANCE_MAP = {
    "exact": 1.0,
    "substitute": 0.1,
    "complement": 0.01,
    "irrelevant": 0.0
}

# Relevance scores for raw ESCI codes (used in test_df_filtered ground truth)
ESCI_CODE_MAP = {
    "E": 1.0,
    "S": 0.1,
    "C": 0.01,
    "I": 0.0
}

def get_ndcg(hits, query_id, k=5):
    actual_relevance = [RELEVANCE_MAP.get(h['label'].lower(), 0.0) for h in hits[:k]]
    dcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(actual_relevance))

    # IDCG: use raw ESCI codes from test_df_filtered
    query_gt = test_df_filtered[test_df_filtered['query_id'] == query_id]
    ideal_relevance = sorted(
        [ESCI_CODE_MAP.get(l, 0.0) for l in query_gt['esci_label']],
        reverse=True
    )
    idcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(ideal_relevance[:k]))

    return dcg / idcg if idcg > 0 else 0.0

In [169]:
# Aggregate nDCG@K scores across all sampled queries
K = 5
all_ndcg_scores = []

for result, (_, eval_row) in zip(evaluation_results, eval_queries.iterrows()):
    q_id = eval_row['query_id']
    score = get_ndcg(result['hits'], q_id, k=K)
    all_ndcg_scores.append(score)

mean_ndcg = np.mean(all_ndcg_scores)
print(f"--- Final Evaluation Results ---")
print(f"Mean nDCG@{K}: {mean_ndcg:.4f}")

--- Final Evaluation Results ---
Mean nDCG@5: 0.5684
